# Suggesting the Expense Type for Department for Transport spend lines

**Patrick Gilliland**

**Question.** Can a model trained on the Department for Transport's past ledger lines suggest the right Expense Type for a new line over £25,000, and what share of lines could it code confidently enough to leave the rest to a person?

**Task 1: the problem and the dataset.** Every payment line a finance team posts carries an account code, which in this data is the `Expense Type`. The problem is to suggest that code for a new line from the other fields on the line, such as the supplier and the item text. In accounting terms this is general ledger account classification: assigning an account code to each posting line. The task is supervised learning (classification), and it is multi-class, because `Expense Type` holds many labels. The dataset is [Spend over £25,000 in the Department for Transport and its arms-length bodies](https://www.data.gov.uk/dataset/804be0c6-ae01-4899-bbb9-2317dff1a4f9/financial-transactions-data-dft), which the Department for Transport publishes on data.gov.uk as one CSV file a month. I use the 48 months from April 2022 to March 2026. `build_dataset.py` stacks the original monthly files in `dft_originals/` into `dft_spend_over_25k_2022-04_to_2026-03.csv`, beside this notebook, without changing any cell. Contains public sector information licensed under the Open Government Licence v3.0.

**Report.** The Machine Learning Analysis Report is one PDF under the two names the brief uses: `module_summary.pdf`, and an identical copy named `Machine_Learning_Analysis_Report.pdf`.

## Data preparation

### Setup

In [1]:
import hashlib
import os
import platform
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import sklearn

# I show every column and whole cell text, and figures with two decimal places
# and thousands separators, so the wide tables below read without scientific
# notation.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# Every step below that draws at random takes this seed, so a rerun gives the
# same numbers.
SEED = 42

print(f"Python {platform.python_version()}")
print(f"NumPy {np.__version__}")
print(f"pandas {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"Matplotlib {matplotlib.__version__}")
print(f"Random seed: {SEED}")
print(f"CPU cores: {os.cpu_count()}")

Python 3.12.3
NumPy 2.5.3
pandas 3.0.6
scikit-learn 1.9.1
Matplotlib 3.11.2
Random seed: 42
CPU cores: 24


**Hardware.** I run everything on the CPU, with no GPU. The models are scikit-learn's, and the whole file fits in memory, so I load it in one go. Where a scikit-learn step can use every core, it gets `n_jobs=-1`.

### Task 2: Load and inspect the dataset

In [2]:
DATA = Path("dft_spend_over_25k_2022-04_to_2026-03.csv")
EXPECTED_SHA256 = (
    "3d9d7f911ef00f9394c07077ebb1ec161efbeb7f3073824c801a662aac212bd4"
)

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} is not beside this notebook. Build it from the 48 original "
        "monthly files in dft_originals/ with: python build_dataset.py"
    )
digest = hashlib.sha256(DATA.read_bytes()).hexdigest()
print(f"File: {DATA}, {DATA.stat().st_size:,} bytes")
print(f"SHA-256: {digest}")
# Every figure below comes from this exact file, so I stop rather than
# analyse a different one.
if digest != EXPECTED_SHA256:
    raise ValueError(
        f"{DATA} is not the file build_dataset.py writes from the 48 original "
        "files. Rebuild it with: python build_dataset.py"
    )
print(
    "It matches the file build_dataset.py writes from the 48 original files."
)

# I read every cell as the text DfT published. The amounts carry a pound sign
# and commas, and by default pandas would turn text such as "#N/A" into a
# blank, so nothing is converted on load.
df = pd.read_csv(DATA, dtype=str, keep_default_na=False)
print(f"Loaded: {len(df):,} rows, {df.shape[1]} columns")

File: dft_spend_over_25k_2022-04_to_2026-03.csv, 25,396,696 bytes
SHA-256: 3d9d7f911ef00f9394c07077ebb1ec161efbeb7f3073824c801a662aac212bd4
It matches the file build_dataset.py writes from the 48 original files.
Loaded: 126,346 rows, 11 columns


In [3]:
df.head()

,Department Family,Entity,Date,Expense Type,Expense Area,Supplier,Transaction No,Item Text,£,Postal Code,source_file
0,Department for Transport,Department for Transport,05/04/2022,Grt Aid to NDPBs,DG Rail Strategy and Services Group,NETWORK RAIL ISTRE (CAPITAL GRANTS),2000000612,NR GRANT APRIL 2 SR 9000946295,"£418,408,000.00",NW1 2DN,dft_2022-04.csv
1,Department for Transport,Department for Transport,29/04/2022,Grt Aid to NDPBs,"DG Roads, Places and Environment",National Highways Limited,2000003150,Grant in Aid NH for May 2022,"£298,000,000.00",B1 1RN,dft_2022-04.csv
2,Department for Transport,Department for Transport,29/04/2022,CurrGrtOth to NDPBs,DG Rail Strategy and Services Group,NETWORK RAIL ISTRE (CAPITAL GRANTS),2000003127,NR Grant - DUE 03/05/2022 - SR 9000953144,"£151,086,319.00",NW1 2DN,dft_2022-04.csv
3,Department for Transport,High Speed Two (HS2) Limited,07/04/2022,AUC - Phase 1,000000 AUC - Phase 1,BALFOUR BEATTY VINCI JV - HS2 (N2),#,MWCC Stage One TP,"£135,660,641.62",#,dft_2022-04.csv
4,Department for Transport,Department for Transport,05/04/2022,Cap Grt NDPBs & CG,DG Rail Strategy and Services Group,NETWORK RAIL ISTRE (CAPITAL GRANTS),2000000612,NR GRANT APRIL 2 SR 9000946295,"£117,640,000.00",NW1 2DN,dft_2022-04.csv


In [4]:
published = [name for name in df.columns if name != "source_file"]
stripped = df.apply(lambda column: column.str.strip())
blank_row = stripped[published].eq("").all(axis=1)
blank_files = ", ".join(df.loc[blank_row, "source_file"].unique())
print(
    f"Rows: {len(df):,}, of which {blank_row.sum():,} have every published "
    f"cell blank, all from {blank_files}"
)

# The publisher marks a missing value with "#", and a few cells hold the
# spreadsheet error "#N/A", so I count those beside the truly blank cells.
column_check = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "blank": stripped.eq("").sum(),
        '"#"': stripped.eq("#").sum(),
        '"#N/A" or "N/A"': stripped.isin(["#N/A", "N/A"]).sum(),
        "spaces at either end": df.ne(stripped).sum(),
        "distinct values": df.where(stripped.ne("")).nunique(),
    }
)
column_check

Rows: 126,346, of which 657 have every published cell blank, all from dft_2022-05.csv


,dtype,blank,"""#""","""#N/A"" or ""N/A""",spaces at either end,distinct values
Department Family,str,657,0,0,137,3
Entity,str,657,0,0,1260,15
Date,str,657,0,0,1,1387
Expense Type,str,657,0,0,666,1054
Expense Area,str,657,0,4,756,976
Supplier,str,657,0,0,381,4015
Transaction No,str,657,10192,2,0,97175
Item Text,str,657,16139,57,1977,20231
£,str,657,0,0,50587,104351
Postal Code,str,657,16978,0,210,2791


In [5]:
def parse_fields(frame, amount="£", date="Date", source="source_file"):
    """Return a copy of frame with the amount, date and month as typed columns.

    Three columns are added. amount is the amount in pounds as a float,
    read from text such as " £1,234.50 ". date is the payment date, read as
    day/month/year, or as day.month.year where the file uses dots. month is
    the month of the file the row came from, "YYYY-MM", taken from the
    source file's name. A value that cannot be read is left blank (NaN or
    NaT). frame is not changed.

    Parameters
    ----------
    frame : pandas.DataFrame
        The rows as published, every column text.
    amount, date, source : str
        The names of the amount, date and source file columns.

    Returns
    -------
    pandas.DataFrame
        frame's columns unchanged, then amount, date and month.
    """
    out = frame.copy()
    money = frame[amount].str.strip().str.replace("£", "", regex=False)
    money = money.str.replace(",", "", regex=False)
    out["amount"] = pd.to_numeric(money, errors="coerce").astype(float)
    text = frame[date].str.strip()
    slashes = pd.to_datetime(text, format="%d/%m/%Y", errors="coerce")
    dots = pd.to_datetime(text, format="%d.%m.%Y", errors="coerce")
    out["date"] = slashes.fillna(dots)
    out["month"] = frame[source].str.extract(r"(\d{4}-\d{2})", expand=False)
    return out

In [6]:
df = parse_fields(df)
content = ~blank_row
written = df.loc[content, "Date"]
slash_form = written.str.fullmatch(r"\d{2}/\d{2}/\d{4}")
dot_form = written.str.fullmatch(r"\d{2}\.\d{2}\.\d{4}")
spaced = written.ne(written.str.strip())
print(
    f"Amounts read: {df.loc[content, 'amount'].notna().sum():,} of "
    f"{content.sum():,} lines with content"
)
print(
    f"Dates read: {df.loc[content, 'date'].notna().sum():,} of "
    f"{content.sum():,}; {slash_form.sum():,} written dd/mm/yyyy, "
    f"{dot_form.sum():,} written dd.mm.yyyy, {spaced.sum():,} with spaces "
    "round it"
)
months = df["month"]
print(f"Months: {months.nunique()}, {months.min()} to {months.max()}")
print(f"Published total: £{df['amount'].sum():,.2f}")
print(
    f"Smallest line: £{df['amount'].min():,.2f}; "
    f"largest: £{df['amount'].max():,.2f}"
)
df[["amount", "date", "month"]].dtypes

Amounts read: 125,689 of 125,689 lines with content
Dates read: 125,689 of 125,689; 125,660 written dd/mm/yyyy, 28 written dd.mm.yyyy, 1 with spaces round it
Months: 48, 2022-04 to 2026-03
Published total: £177,111,208,418.51
Smallest line: £25,000.00; largest: £786,575,788.00


amount           float64
date      datetime64[us]
month                str
dtype: object

**What loaded.** The file loads as 126,346 rows and 11 columns: the ten columns DfT publishes and `source_file`, which names the monthly file each row came from. I read every cell as text, so every column loads as text. `parse_fields` then adds the amount as a number, the date as a date and the file's month. Every one of the 125,689 lines with content has an amount and a date I can read, across 48 months.

**Obvious data quality issues.**

- 657 rows are blank in every published column, all from the May 2022 file. The build keeps them, so the combined file holds every row DfT published.
- The amount is text with a pound sign and commas, and 50,587 amount cells have spaces round them. Every amount reads, from £25,000.00 to £786,575,788.00, so no line is a credit.
- The date is text too. 125,660 lines write it dd/mm/yyyy, 28 use dots and 1 has spaces round it. `parse_fields` reads all three forms.
- The publisher writes "#" for a missing value: in 10,192 transaction numbers, 16,139 item texts and 16,978 postcodes. The spreadsheet error "#N/A" or "N/A" sits in 4 Expense Area cells, 2 transaction numbers and 57 item texts. I read the file with `keep_default_na=False`, so pandas cannot turn any of these into a blank without my knowing.
- Names are not always written one way. Department Family has 3 spellings for one department, Entity has 15 values, with spaces round 1,260 of its cells, and Expense Type has 1,054 labels, some of which differ only in case, spacing or punctuation.

I list each of these, and the other anomalies I found, with its count and value in the anomaly list below, before anything is cleaned.

## Modeling

## Evaluation

## Outputs

## Summary